# Improved Main-Class Pipeline V2

Research-only. Non-commercial use only. Not for diagnosis, treatment, or clinical deployment. Doctor review required.

This notebook upgrades the Main_class pipeline in four ordered steps:
1. **Imbalance fix** — WeightedRandomSampler + class-weighted CE with label_smoothing=0.1
2. **Stronger backbone** — ConvNeXt-Tiny trained under the same recipe as EfficientNet-B0 V2
3. **Calibration** — temperature scaling fit on validation logits only
4. **Calibrated top-3 inference** — calibrated shortlist with uncertainty flag instead of raw top-1

Both EfficientNet-B0 and ConvNeXt-Tiny are trained and evaluated; the best backbone is selected using macro_f1, balanced_accuracy, and top3_accuracy.

## 1. Runtime

In Colab: `Runtime` -> `Change runtime type` -> choose GPU (T4 or better).

In [1]:
!nvidia-smi

Mon May 25 23:28:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Drive And Set Constants

Dataset source:

```text
https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/W7OUZM
DOI: 10.7910/DVN/W7OUZM
```

Drive is used to persist downloaded data and training outputs across Colab sessions.
This V2 notebook writes to `outputs_v2/` to avoid overwriting the V1 baseline outputs.

Recommended Drive layout after download:

```text
MyDrive/derm-opd-triage/data/raw/DATASET/*.jpg
MyDrive/derm-opd-triage/data/raw/METADATA/Skin_Metadata.csv
MyDrive/derm-opd-triage/outputs_v2/outputs_v2_effnet/
MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext/
```

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/derm-opd-triage'
REPO_URL = 'https://github.com/Abhigyan-Shekhar/skin-lesion-detect-model.git'
REPO_DIR = '/content/skin-lesion-detect-model'
DATAVERSE_PID = 'doi:10.7910/DVN/W7OUZM'

# Separate output dirs for each backbone
EFFNET_OUTPUT_DIR = 'outputs_v2_effnet'
CONVNEXT_OUTPUT_DIR = 'outputs_v2_convnext'

from pathlib import Path
Path(DRIVE_PROJECT).mkdir(parents=True, exist_ok=True)
(Path(DRIVE_PROJECT) / 'data' / 'raw').mkdir(parents=True, exist_ok=True)
(Path(DRIVE_PROJECT) / 'outputs_v2').mkdir(parents=True, exist_ok=True)
print(f'Drive project folder ready: {DRIVE_PROJECT}')

Mounted at /content/drive
Drive project folder ready: /content/drive/MyDrive/derm-opd-triage


## 3. Clone Repo And Install Dependencies

In [3]:
import os
import subprocess
from pathlib import Path
def run_checked(cmd, *, cwd=None):
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

if not Path(REPO_DIR).exists():
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            'GitHub clone failed. Make the repository public while training or clone it manually from an authenticated Colab session. '
            'Do not paste credentials into notebook cells.'
        )
else:
    run_checked(['git', 'pull'], cwd=REPO_DIR)

%cd {REPO_DIR}
if not Path('requirements.txt').exists():
    raise FileNotFoundError(f'requirements.txt not found in {Path.cwd()}')
!pip install -q -r requirements.txt

No GITHUB_TOKEN found. Clone will only work if the repo is public.
/content/skin-lesion-detect-model
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 162.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 196.7 MB/s eta 0:00:00 0:00:01


## 4. Download Or Link DermaCon-IN Data

The notebook first checks `MyDrive/derm-opd-triage/data/raw`. If that cache is empty, it downloads files from Harvard Dataverse using DOI `10.7910/DVN/W7OUZM`, then symlinks the Drive data into the repo.

If Dataverse throttles large downloads, manually place the files in the Drive layout and rerun this cell.

In [4]:
from pathlib import Path
import shutil

drive_raw = Path(DRIVE_PROJECT) / 'data' / 'raw'
repo_data = Path(REPO_DIR) / 'data'
repo_raw = repo_data / 'raw'
repo_splits = repo_data / 'splits'
repo_data.mkdir(parents=True, exist_ok=True)
repo_splits.mkdir(parents=True, exist_ok=True)

metadata_path = drive_raw / 'METADATA' / 'Skin_Metadata.csv'
image_dir = drive_raw / 'DATASET'
image_count = sum(1 for path in image_dir.rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}) if image_dir.exists() else 0

if metadata_path.exists() and image_count > 0:
    print('Using existing Drive dataset cache:', drive_raw)
    print('Images found:', image_count)
else:
    print('Drive dataset cache is incomplete. Downloading DermaCon-IN from Harvard Dataverse...')
    drive_raw.mkdir(parents=True, exist_ok=True)
    run_checked(['python', 'src/download_dataverse.py', '--persistent_id', DATAVERSE_PID, '--output_dir', str(drive_raw)])
    image_count = sum(1 for path in image_dir.rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}) if image_dir.exists() else 0

if not metadata_path.exists():
    raise FileNotFoundError(
        f'Metadata missing after download: {metadata_path}. '
        'If Dataverse returned 403, open the dataset page in a browser and manually upload Skin_Metadata.csv to this Drive path.'
    )
if image_count == 0:
    raise FileNotFoundError(
        f'No images found after download: {image_dir}. '
        'If Dataverse blocks bulk download, manually upload the .jpg/.png files to this Drive folder.'
    )

if repo_raw.exists() or repo_raw.is_symlink():
    if repo_raw.is_symlink():
        repo_raw.unlink()
    else:
        shutil.rmtree(repo_raw)

repo_raw.symlink_to(drive_raw, target_is_directory=True)
print('Linked', repo_raw, '->', drive_raw)
!find data/raw -maxdepth 3 -type f | head -40

Using existing Drive dataset cache: /content/drive/MyDrive/derm-opd-triage/data/raw
Images found: 5450
Linked /content/skin-lesion-detect-model/data/raw -> /content/drive/MyDrive/derm-opd-triage/data/raw


## 5. Inspect Metadata

In [5]:
!python src/inspect_metadata.py \
  --metadata data/raw/METADATA/Skin_Metadata.csv \
  --image_dir data/raw/DATASET


=== Metadata Inspection ===
Rows: 5450
Image directory: data/raw/DATASET

Available columns:
- Age
- Body_part
- Descriptors
- Confidence
- Disease_label
- Fitzpatrick
- Gradability
- Image_name
- Main_class
- Monk_skin_tone
- Quality
- Sex
- Sub_class
- Subject_ID

Detected columns:
- image: Image_name
- diagnosis: None
- main_class: Main_class
- subclass: Sub_class
- age: Age
- sex: Sex
- patient_id: Subject_ID
- body_site: None
- fitzpatrick: Fitzpatrick
- monk_skin_tone: Monk_skin_tone
- lesion_concepts: []

Missing value percentage:
- Age: 0.0%
- Body_part: 0.0%
- Descriptors: 0.0%
- Confidence: 0.0%
- Disease_label: 0.0%
- Fitzpatrick: 0.11%
- Gradability: 0.0%
- Image_name: 0.0%
- Main_class: 0.0%
- Monk_skin_tone: 0.11%
- Quality: 0.0%
- Sex: 0.0%
- Sub_class: 0.0%
- Subject_ID: 0.0%

Unique patients: 3002
Unique image references: 5450
Existing image files: 5450
Missing image files: 0

Label distribution for main_class:
- Infectious Disorders: 2227
- Inflammatory Disorders: 15

## 6. Prepare Splits

If patient IDs exist, splitting is patient-level. If official `train_split.csv` and `test_split.csv` exist under `data/raw`, they are used.

In [6]:
!python src/prepare_splits.py \
  --metadata data/raw/METADATA/Skin_Metadata.csv \
  --output_dir data/splits

!cat data/splits/split_summary.json


=== Split Summary ===
- split_method: official_train_test_plus_val_from_train
- train_rows: 3630
- val_rows: 769
- test_rows: 1051
- label_column: Main_class
- detected_columns: {'image': 'Image_name', 'diagnosis': None, 'main_class': 'Main_class', 'subclass': 'Sub_class', 'age': 'Age', 'sex': 'Sex', 'patient_id': 'Subject_ID', 'body_site': None, 'fitzpatrick': 'Fitzpatrick', 'monk_skin_tone': 'Monk_skin_tone', 'lesion_concepts': []}

Saved splits to: data/splits
{
  "split_method": "official_train_test_plus_val_from_train",
  "train_rows": 3630,
  "val_rows": 769,
  "test_rows": 1051,
  "label_column": "Main_class",
  "detected_columns": {
    "image": "Image_name",
    "diagnosis": null,
    "main_class": "Main_class",
    "subclass": "Sub_class",
    "age": "Age",
    "sex": "Sex",
    "patient_id": "Subject_ID",
    "body_site": null,
    "fitzpatrick": "Fitzpatrick",
    "monk_skin_tone": "Monk_skin_tone",
    "lesion_concepts": []
  }
}

## 7. Imports And Setup

All third-party and local imports are collected here so that any missing dependency is caught before the long training run begins.

In [7]:
from __future__ import annotations

import csv
import json
import shutil
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    auc,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import LBFGS, AdamW
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm

sys.path.insert(0, 'src')
from dataset import DermatologyDataset, build_transforms
from metrics import build_classification_report, compute_epoch_metrics
from models import build_model, unfreeze_last_blocks
from train import run_epoch
from utils import (
    DISCLAIMER_TEXT,
    choose_label_column,
    detect_columns,
    ensure_dir,
    read_table,
    resolve_device,
    set_seed,
    write_json,
)

DATA_TRAIN_CSV = 'data/splits/train.csv'
DATA_VAL_CSV   = 'data/splits/val.csv'
DATA_TEST_CSV  = 'data/splits/test.csv'
DATA_IMAGE_DIR = 'data/raw/DATASET'

print('Imports OK')
print('CUDA available:', torch.cuda.is_available())

Imports OK
CUDA available: True


## 8. Helper Functions

- **`build_weighted_sampler`** — assigns each sample a weight proportional to the inverse of its class frequency so that minority classes appear as often as majority classes during training.
- **`make_loss_v2`** — constructs a class-weighted `CrossEntropyLoss` with `label_smoothing=0.1` to reduce overconfidence.
- **`TemperatureScaler`** — a single learnable parameter that divides logits, shifting the softmax distribution without changing relative ordering.
- **`collect_val_logits`** — runs the model over the validation set in no-grad mode and collects raw logits for calibration.
- **`calibrate_temperature`** — fits the temperature parameter on validation logits via LBFGS to minimise NLL.
- **`evaluate_full`** — full test-set evaluation: calibrated metrics, per-class breakdown, confusion matrices, ROC curves, top-3 CSV.
- **`predict_top3_calibrated`** — single-image calibrated top-3 inference with uncertainty flag.
- **`train_v2`** — two-phase training (head + finetune) with `WeightedRandomSampler` and `label_smoothing`.

In [8]:
def build_weighted_sampler(train_dataset: DermatologyDataset) -> WeightedRandomSampler:
    """Assign each sample weight = 1 / class_count so minority classes appear more often."""
    labels = train_dataset.df[train_dataset.label_column].astype(str).tolist()
    label_indices = [train_dataset.class_to_idx[l] for l in labels]
    class_counts = Counter(label_indices)
    sample_weights = [1.0 / class_counts[idx] for idx in label_indices]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

In [9]:
def make_loss_v2(train_dataset: DermatologyDataset, device: torch.device, label_smoothing: float = 0.1) -> nn.Module:
    """Class-weighted CrossEntropyLoss with label smoothing (training only)."""
    labels = train_dataset.df[train_dataset.label_column].astype(str).tolist()
    classes = np.array(sorted(train_dataset.class_to_idx.keys()))
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    tensor_weights = torch.tensor(weights, dtype=torch.float32, device=device)
    return nn.CrossEntropyLoss(weight=tensor_weights, label_smoothing=label_smoothing)

In [10]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature.clamp(min=0.05)


def collect_val_logits(
    model: nn.Module,
    val_loader: DataLoader,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Collect raw logits and ground-truth labels from validation set (no gradients)."""
    model.eval()
    all_logits: list[torch.Tensor] = []
    all_targets: list[int] = []
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc='Collecting val logits', leave=False):
            images = images.to(device)
            logits = model(images).cpu()
            all_logits.append(logits)
            all_targets.extend(labels.tolist())
    return torch.cat(all_logits, dim=0), torch.tensor(all_targets)


def calibrate_temperature(
    logits: torch.Tensor,
    targets: torch.Tensor,
    output_dir: str | Path | None = None,
) -> tuple[float, TemperatureScaler]:
    """Fit a single temperature parameter on validation logits using LBFGS.

    Saves temperature.json to output_dir/metrics/ if provided.
    Returns (temperature_float, fitted_scaler).
    """
    scaler = TemperatureScaler()
    optimizer = LBFGS([scaler.temperature], lr=0.01, max_iter=100)
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(scaler(logits), targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    temperature = float(scaler.temperature.item())
    print(f'Fitted temperature: {temperature:.4f}')

    if output_dir is not None:
        write_json(
            Path(output_dir) / 'metrics' / 'temperature.json',
            {'temperature': temperature},
        )

    return temperature, scaler

In [11]:
def evaluate_full(
    model: nn.Module,
    test_loader: DataLoader,
    class_names: list[str],
    image_column: str,
    temperature: float,
    device: torch.device,
    output_dir: str | Path,
    tag: str = '',
) -> dict:
    """Run full evaluation: standard metrics, ROC-AUC, both confusion matrices, ROC curves, top-3 predictions CSV."""
    output_dir = Path(output_dir)
    metrics_dir = ensure_dir(output_dir / 'metrics')
    plots_dir = ensure_dir(output_dir / 'plots')
    predictions_dir = ensure_dir(output_dir / 'predictions')
    n_classes = len(class_names)
    suffix = f'_{tag}' if tag else ''

    # --- Forward pass ---
    model.eval()
    all_logits: list[torch.Tensor] = []
    all_targets: list[int] = []
    all_image_ids: list[str] = []

    with torch.no_grad():
        for images, labels, metadata in tqdm(test_loader, desc=f'Evaluating{" " + tag if tag else ""}', leave=False):
            images = images.to(device)
            logits = model(images).cpu()
            all_logits.append(logits)
            all_targets.extend(labels.tolist())
            batch_n = len(labels)
            ids = metadata.get(image_column, [str(i) for i in range(batch_n)])
            all_image_ids.extend(str(ids[i]) for i in range(batch_n))

    logits_t = torch.cat(all_logits, dim=0)
    targets_np = np.array(all_targets)
    targets_t = torch.tensor(targets_np)
    T = max(float(temperature), 0.05)

    # --- Probabilities ---
    raw_probs = torch.softmax(logits_t, dim=1).numpy()
    cal_probs = torch.softmax(logits_t / T, dim=1).numpy()
    cal_preds = cal_probs.argmax(axis=1)

    # --- Top-k accuracy helper ---
    cal_probs_t = torch.tensor(cal_probs)

    def topk_acc(k: int) -> float:
        k_actual = min(k, n_classes)
        topk = torch.topk(cal_probs_t, k=k_actual, dim=1).indices
        return float(topk.eq(targets_t.unsqueeze(1)).any(dim=1).float().mean().item())

    # --- Standard metrics ---
    metrics: dict = {
        'tag': tag or 'default',
        'temperature': T,
        'accuracy': float(accuracy_score(targets_np, cal_preds)),
        'balanced_accuracy': float(balanced_accuracy_score(targets_np, cal_preds)),
        'macro_f1': float(f1_score(targets_np, cal_preds, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(targets_np, cal_preds, average='weighted', zero_division=0)),
        'top3_accuracy': topk_acc(3),
        'top5_accuracy': topk_acc(5),
        'calibrated_nll': float(F.cross_entropy(logits_t / T, targets_t).item()),
        'raw_nll': float(F.cross_entropy(logits_t, targets_t).item()),
        'disclaimer': DISCLAIMER_TEXT,
    }

    # --- ROC-AUC (multiclass OVR) ---
    try:
        metrics['macro_roc_auc'] = float(roc_auc_score(targets_np, cal_probs, multi_class='ovr', average='macro'))
        metrics['weighted_roc_auc'] = float(roc_auc_score(targets_np, cal_probs, multi_class='ovr', average='weighted'))
    except Exception as exc:
        print(f'  [warn] Multiclass ROC-AUC failed: {exc}')
        metrics['macro_roc_auc'] = None
        metrics['weighted_roc_auc'] = None

    # --- Per-class ROC-AUC ---
    per_class_roc_auc = {}
    for i, cls_name in enumerate(class_names):
        binary_y = (targets_np == i).astype(int)
        if binary_y.sum() >= 2 and (1 - binary_y).sum() >= 2:
            try:
                per_class_roc_auc[cls_name] = float(roc_auc_score(binary_y, cal_probs[:, i]))
            except Exception:
                per_class_roc_auc[cls_name] = None
        else:
            per_class_roc_auc[cls_name] = None
    metrics['per_class_roc_auc'] = per_class_roc_auc

    # --- Write metrics ---
    write_json(metrics_dir / f'metrics{suffix}.json', metrics)

    precision, recall, f1_vals, support = precision_recall_fscore_support(
        targets_np, cal_preds, labels=list(range(n_classes)), zero_division=0
    )
    per_class_df = pd.DataFrame({
        'class_name': class_names,
        'precision': precision,
        'recall': recall,
        'f1': f1_vals,
        'support': support,
        'roc_auc': [per_class_roc_auc.get(c) for c in class_names],
    })
    per_class_df.to_csv(metrics_dir / f'per_class_metrics{suffix}.csv', index=False)

    report = build_classification_report(targets_np.tolist(), cal_preds.tolist(), class_names)
    with open(metrics_dir / f'classification_report{suffix}.txt', 'w', encoding='utf-8') as fh:
        fh.write(report)
        fh.write(f'\n\n{DISCLAIMER_TEXT}\n')

    # --- Calibration summary ---
    write_json(metrics_dir / f'calibration_summary{suffix}.json', {
        'temperature': T,
        'calibrated_nll': metrics['calibrated_nll'],
        'raw_nll': metrics['raw_nll'],
        'nll_improvement': metrics['raw_nll'] - metrics['calibrated_nll'],
    })

    # --- Top-3 predictions CSV ---
    rows = []
    for i in range(len(targets_np)):
        top3_idx = np.argsort(cal_probs[i])[::-1][:min(3, n_classes)]
        top1_p = float(cal_probs[i][top3_idx[0]])
        top2_p = float(cal_probs[i][top3_idx[1]]) if len(top3_idx) >= 2 else 0.0
        rows.append({
            'image_id': all_image_ids[i],
            'true_label': class_names[targets_np[i]],
            'top1_label': class_names[top3_idx[0]],
            'top1_prob': round(top1_p, 6),
            'top2_label': class_names[top3_idx[1]] if len(top3_idx) >= 2 else '',
            'top2_prob': round(top2_p, 6),
            'top3_label': class_names[top3_idx[2]] if len(top3_idx) >= 3 else '',
            'top3_prob': round(float(cal_probs[i][top3_idx[2]]), 6) if len(top3_idx) >= 3 else 0.0,
            'uncertainty_flag': bool(top1_p < 0.45 or (top1_p - top2_p) < 0.10),
        })
    pd.DataFrame(rows).to_csv(predictions_dir / f'predictions_top3{suffix}.csv', index=False)

    # --- Confusion matrices ---
    cm_counts = confusion_matrix(targets_np, cal_preds, labels=list(range(n_classes)))
    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts.astype(float) / np.where(row_sums == 0, 1, row_sums)

    assert np.allclose(cm_norm.sum(axis=1), 1.0, atol=1e-6), 'Normalized CM rows do not sum to 1'

    fig_w = max(8, n_classes * 0.6)

    fig, ax = plt.subplots(figsize=(fig_w, fig_w))
    im = ax.imshow(cm_counts, cmap='Blues')
    ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
    ax.set_xticklabels(class_names, rotation=90, fontsize=max(5, 8 - n_classes // 10))
    ax.set_yticklabels(class_names, fontsize=max(5, 8 - n_classes // 10))
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix \u2014 Counts{" (" + tag + ")" if tag else ""}')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(plots_dir / f'confusion_matrix_counts{suffix}.png', dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(fig_w, fig_w))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
    ax.set_xticklabels(class_names, rotation=90, fontsize=max(5, 8 - n_classes // 10))
    ax.set_yticklabels(class_names, fontsize=max(5, 8 - n_classes // 10))
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix \u2014 Row-Normalized{" (" + tag + ")" if tag else ""}')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(plots_dir / f'confusion_matrix_normalized{suffix}.png', dpi=150)
    plt.close(fig)

    # --- ROC curves ---
    fig, ax = plt.subplots(figsize=(10, 8))
    macro_tpr = np.zeros(100)
    mean_fpr = np.linspace(0, 1, 100)
    valid_count = 0
    for i, cls_name in enumerate(class_names):
        binary_y = (targets_np == i).astype(int)
        if binary_y.sum() < 2 or (1 - binary_y).sum() < 2:
            continue
        try:
            fpr, tpr, _ = roc_curve(binary_y, cal_probs[:, i])
            cls_auc = auc(fpr, tpr)
            macro_tpr += np.interp(mean_fpr, fpr, tpr)
            valid_count += 1
            ax.plot(fpr, tpr, alpha=0.3, lw=1, label=f'{cls_name} ({cls_auc:.2f})')
        except Exception:
            pass
    if valid_count > 0:
        macro_tpr /= valid_count
        macro_auc_val = auc(mean_fpr, macro_tpr)
        ax.plot(mean_fpr, macro_tpr, color='navy', lw=2, label=f'Macro avg ({macro_auc_val:.2f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves (One-vs-Rest)' + (' — ' + tag if tag else ''))
    ax.legend(loc='lower right', fontsize=6, ncol=2)
    fig.tight_layout()
    fig.savefig(plots_dir / f'roc_curves{suffix}.png', dpi=150)
    plt.close(fig)

    # --- Print summary ---
    print(f'\n=== Evaluation{" [" + tag + "]" if tag else ""} ===')
    for key in ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'top3_accuracy', 'macro_roc_auc', 'calibrated_nll']:
        val = metrics.get(key)
        print(f'  {key}: {val:.4f}' if isinstance(val, float) else f'  {key}: {val}')
    print(f'  raw_nll \u2192 calibrated_nll: {metrics["raw_nll"]:.4f} \u2192 {metrics["calibrated_nll"]:.4f}')
    print(f'  Saved plots and metrics to: {output_dir}')

    return metrics

In [12]:
def predict_top3_calibrated(
    bundle: dict,
    temperature: float,
    image_path: str | Path,
) -> dict:
    """Calibrated top-3 inference with uncertainty flag.

    uncertainty_flag is True when:
      - calibrated top-1 probability < 0.45, OR
      - (top-1 prob - top-2 prob) < 0.10
    """
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f'Image not found: {image_path}')

    image = Image.open(image_path).convert('RGB')
    tensor = bundle['transform'](image).unsqueeze(0).to(bundle['device'])
    T = max(float(temperature), 0.05)

    with torch.no_grad():
        logits = bundle['model'](tensor)
        cal_probs = torch.softmax(logits / T, dim=1)[0]
        k = min(3, cal_probs.shape[0])
        values, indices = torch.topk(cal_probs, k=k)

    predictions = [
        {'label': bundle['idx_to_class'][int(idx.item())], 'probability': round(float(v.item()), 6)}
        for v, idx in zip(values, indices)
    ]

    top1_p = float(values[0].item()) if len(values) >= 1 else 0.0
    top2_p = float(values[1].item()) if len(values) >= 2 else 0.0
    top1_top2_margin = top1_p - top2_p
    uncertainty_flag = bool(top1_p < 0.45 or top1_top2_margin < 0.10)
    conf_level = 'high' if top1_p >= 0.75 else ('moderate' if top1_p >= 0.45 else 'low')

    return {
        'top_predictions': predictions,
        'confidence_level': conf_level,
        'uncertainty_flag': uncertainty_flag,
        'top1_top2_margin': round(top1_top2_margin, 6),
        'temperature': T,
        'model_name': bundle['config']['model_name'],
        'task_name': bundle['config'].get('task_name'),
        'checkpoint_path': bundle['checkpoint_path'],
        'disclaimer': DISCLAIMER_TEXT,
    }

In [13]:
def train_v2(
    config: dict,
    output_dir: str | Path,
    train_csv: str | Path = DATA_TRAIN_CSV,
    val_csv: str | Path = DATA_VAL_CSV,
    label_column_override: str | None = None,
) -> dict:
    """Two-phase training with WeightedRandomSampler + label_smoothing.

    Returns a bundle dict compatible with predict_top3_calibrated and evaluate_full.
    """
    output_dir = Path(output_dir)
    set_seed(int(config.get('seed', 42)))

    # --- Detect columns ---
    train_df = read_table(Path(train_csv))
    detected = detect_columns(list(train_df.columns))
    image_column = detected.get('image')
    if not image_column:
        raise ValueError(f'Cannot detect image column from columns: {list(train_df.columns)}')
    label_column = choose_label_column(detected, label_column_override)
    if not label_column or label_column not in train_df.columns:
        raise ValueError(
            f'Cannot detect label column. Pass label_column_override=... '
            f'Available columns: {list(train_df.columns)}'
        )

    class_names = sorted(train_df[label_column].dropna().astype(str).unique().tolist())
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    idx_to_class = {idx: name for name, idx in class_to_idx.items()}
    n_classes = len(class_names)
    print(f'Classes ({n_classes}): {class_names}')

    device = resolve_device(config.get('device', 'auto'))
    amp_enabled = bool(config.get('amp', True)) and device.type == 'cuda'
    image_size = int(config.get('image_size', 224))

    # --- Datasets ---
    train_dataset = DermatologyDataset(
        csv_path=train_csv, image_dir=config['image_dir'],
        image_column=image_column, label_column=label_column,
        class_to_idx=class_to_idx,
        transform=build_transforms(image_size=image_size, train=True),
    )
    val_dataset = DermatologyDataset(
        csv_path=val_csv, image_dir=config['image_dir'],
        image_column=image_column, label_column=label_column,
        class_to_idx=class_to_idx,
        transform=build_transforms(image_size=image_size, train=False),
    )

    loader_kwargs = {
        'batch_size': int(config.get('batch_size', 32)),
        'num_workers': int(config.get('num_workers', 2)),
        'pin_memory': bool(config.get('pin_memory', True)),
    }

    use_sampler = config.get('sampler', 'none') == 'weighted_random'
    if use_sampler:
        sampler = build_weighted_sampler(train_dataset)
        train_loader = DataLoader(
            train_dataset, sampler=sampler,
            drop_last=bool(config.get('drop_last', False)), **loader_kwargs,
        )
        print(f'WeightedRandomSampler active for {len(train_dataset)} train samples')
    else:
        train_loader = DataLoader(
            train_dataset, shuffle=True,
            drop_last=bool(config.get('drop_last', False)), **loader_kwargs,
        )

    val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)

    # --- Model and loss ---
    model = build_model(
        model_name=config['model_name'],
        num_classes=n_classes,
        freeze_backbone=bool(config.get('freeze_backbone', True)),
    )
    model = model.to(device)

    criterion = make_loss_v2(
        train_dataset, device,
        label_smoothing=float(config.get('label_smoothing', 0.1)),
    )
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    checkpoints_dir = ensure_dir(output_dir / 'checkpoints')
    metrics_dir = ensure_dir(output_dir / 'metrics')
    history: list[dict] = []
    best_macro_f1 = -1.0
    patience = int(config.get('early_stopping_patience', 7))
    patience_counter = 0
    global_epoch = 0
    stopped_early = False

    training_phases = [
        {
            'name': 'head',
            'epochs': int(config.get('epochs_head', 15)),
            'lr': float(config.get('lr_head', 1e-3)),
            'before_phase': None,
        },
        {
            'name': 'finetune',
            'epochs': int(config.get('epochs_finetune', 20)),
            'lr': float(config.get('lr_finetune', 1e-5)),
            'before_phase': lambda: unfreeze_last_blocks(config['model_name'], model),
        },
    ]

    for phase in training_phases:
        if stopped_early:
            break
        if phase['before_phase'] is not None:
            phase['before_phase']()
            patience_counter = 0
            print(f"  Reset early-stopping patience for phase: {phase['name']}")

        optimizer = AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=phase['lr'],
            weight_decay=float(config.get('weight_decay', 1e-4)),
        )

        for phase_epoch in range(1, phase['epochs'] + 1):
            global_epoch += 1
            print(f'\n=== {config["model_name"]} | {phase["name"]} | epoch {phase_epoch}/{phase["epochs"]} (global {global_epoch}) ===')

            train_loss, train_metrics = run_epoch(model, train_loader, criterion, device, optimizer, scaler, amp_enabled)
            val_loss, val_metrics = run_epoch(model, val_loader, criterion, device, None, None, amp_enabled)

            row = {
                'global_epoch': global_epoch, 'phase': phase['name'], 'phase_epoch': phase_epoch,
                'train_loss': round(train_loss, 6), 'val_loss': round(val_loss, 6),
                **{f'train_{k}': round(v, 6) for k, v in train_metrics.items()},
                **{f'val_{k}': round(v, 6) for k, v in val_metrics.items()},
            }
            history.append(row)
            with open(metrics_dir / 'train_history.json', 'w', encoding='utf-8') as fh:
                json.dump(history, fh, indent=2)

            ckpt_payload = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': global_epoch,
                'config': config,
                'class_to_idx': class_to_idx,
                'image_column': image_column,
                'label_column': label_column,
                'best_macro_f1': best_macro_f1,
            }
            torch.save(ckpt_payload, checkpoints_dir / 'latest.pt')

            current_f1 = val_metrics['macro_f1']
            print(f'  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_macro_f1={current_f1:.4f}  best={best_macro_f1:.4f}')

            if current_f1 > best_macro_f1:
                best_macro_f1 = current_f1
                patience_counter = 0
                ckpt_payload['best_macro_f1'] = best_macro_f1
                torch.save(ckpt_payload, checkpoints_dir / 'best.pt')
                print(f'  New best checkpoint saved (macro_f1={best_macro_f1:.4f})')
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f'  Early stopping after {patience_counter} non-improving epochs.')
                    stopped_early = True
                    break

    write_json(metrics_dir / 'training_summary.json', {
        'model_name': config['model_name'],
        'best_macro_f1': best_macro_f1,
        'completed_epochs': global_epoch,
        'stopped_early': stopped_early,
        'label_column': label_column,
        'image_column': image_column,
        'class_names': class_names,
    })
    print(f'\nTraining complete. Best val macro_f1={best_macro_f1:.4f}')
    print(f'Best checkpoint: {checkpoints_dir / "best.pt"}')

    best_ckpt = torch.load(checkpoints_dir / 'best.pt', map_location=device)
    model.load_state_dict(best_ckpt['model_state_dict'])
    model.eval()
    print(
        f"Reloaded best checkpoint from epoch {best_ckpt.get('epoch', '?')} "
        f"with best_macro_f1={best_ckpt.get('best_macro_f1', best_macro_f1):.4f}"
    )
    return {
        'checkpoint_path': str(checkpoints_dir / 'best.pt'),
        'config': config,
        'class_to_idx': class_to_idx,
        'idx_to_class': idx_to_class,
        'image_column': image_column,
        'label_column': label_column,
        'class_names': class_names,
        'device': device,
        'model': model,
        'transform': build_transforms(image_size=image_size, train=False),
        'best_macro_f1': best_macro_f1,
    }

## 9. Step 1 — Train EfficientNet-B0 V2

Changes vs the V1 baseline (`train_effnet_b0_colab.ipynb`):

- **WeightedRandomSampler** (`sampler: weighted_random`) replaces the plain `shuffle=True` DataLoader, ensuring each training batch samples minority classes at the same expected frequency as majority classes.
- **Class-weighted loss with label smoothing 0.1** (`make_loss_v2`) penalises the model more for errors on rare classes and reduces overconfidence.
- Training phases and hyperparameters are otherwise identical to V1 to allow a fair comparison.

In [14]:
EFFNET_V2_CONFIG = {
    'model_name': 'efficientnet_b0',
    'task_name': 'main_class_v2',
    'image_dir': DATA_IMAGE_DIR,
    'sampler': 'weighted_random',
    'use_class_weights': True,
    'label_smoothing': 0.1,
    'epochs_head': 15,
    'epochs_finetune': 20,
    'lr_head': 1e-3,
    'lr_finetune': 1e-5,
    'weight_decay': 1e-4,
    'batch_size': 32,
    'image_size': 224,
    'num_workers': 2,
    'early_stopping_patience': 7,
    'freeze_backbone': True,
    'amp': True,
    'pin_memory': True,
    'drop_last': False,
    'seed': 42,
    'device': 'auto',
}
print('EfficientNet-B0 V2 config ready.')

EfficientNet-B0 V2 config ready.


In [15]:
effnet_bundle = train_v2(EFFNET_V2_CONFIG, EFFNET_OUTPUT_DIR)

Classes (8): ['Infectious Disorders', 'Inflammatory Disorders', 'Keratanisation Disorders', 'Neoplasms and tumors', 'No Definite Diagnosis', 'Other skin disorders', 'Pigmentary Disorders', 'Skin Appendages Disorders']
WeightedRandomSampler active for 3630 train samples
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 344MB/s]
/tmp/ipykernel_1687/3965782088.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)



=== efficientnet_b0 | head | epoch 1/15 (global 1) ===


  train_loss=1.2162  val_loss=3.3885  val_macro_f1=0.0387  best=-1.0000
  New best checkpoint saved (macro_f1=0.0387)

=== efficientnet_b0 | head | epoch 2/15 (global 2) ===


  train_loss=0.8675  val_loss=3.2840  val_macro_f1=0.0622  best=0.0387
  New best checkpoint saved (macro_f1=0.0622)

=== efficientnet_b0 | head | epoch 3/15 (global 3) ===


...

  train_loss=0.5156  val_loss=3.1412  val_macro_f1=0.1666  best=0.1677

=== efficientnet_b0 | finetune | epoch 17/20 (global 32) ===


  train_loss=0.4893  val_loss=3.1581  val_macro_f1=0.1695  best=0.1677
  New best checkpoint saved (macro_f1=0.1695)

=== efficientnet_b0 | finetune | epoch 18/20 (global 33) ===


  train_loss=0.5149  val_loss=3.1465  val_macro_f1=0.1622  best=0.1695

=== efficientnet_b0 | finetune | epoch 19/20 (global 34) ===


  train_loss=0.4889  val_loss=3.1611  val_macro_f1=0.1668  best=0.1695

=== efficientnet_b0 | finetune | epoch 20/20 (global 35) ===


  train_loss=0.4889  val_loss=3.1225  val_macro_f1=0.1614  best=0.1695

Training complete. Best val macro_f1=0.1695
Best checkpoint: outputs_v2_effnet/checkpoints/best.pt


## 10. Step 2 — Calibrate EfficientNet-B0

Temperature scaling fits a single scalar `T` on **validation** logits only (never touches the test set). After calibration, `softmax(logits / T)` produces better-calibrated probabilities — the model's confidence is closer to its actual accuracy. A temperature `T > 1` softens the distribution (reduces overconfidence); `T < 1` sharpens it.

The fitted temperature is saved to `outputs_v2_effnet/metrics/temperature.json` for reproducibility.

In [16]:
# Collect validation logits from best checkpoint
val_dataset_effnet = DermatologyDataset(
    csv_path=DATA_VAL_CSV,
    image_dir=DATA_IMAGE_DIR,
    image_column=effnet_bundle['image_column'],
    label_column=effnet_bundle['label_column'],
    class_to_idx=effnet_bundle['class_to_idx'],
    transform=build_transforms(image_size=224, train=False),
)
val_loader_effnet = DataLoader(val_dataset_effnet, batch_size=32, shuffle=False, num_workers=2)

effnet_val_logits, effnet_val_targets = collect_val_logits(
    effnet_bundle['model'], val_loader_effnet, effnet_bundle['device']
)
effnet_T, effnet_scaler = calibrate_temperature(
    effnet_val_logits, effnet_val_targets, output_dir=EFFNET_OUTPUT_DIR
)
print(f'EfficientNet-B0 temperature: {effnet_T:.4f}')

Fitted temperature: 2.0514
EfficientNet-B0 temperature: 2.0514


## 11. Step 3 — Evaluate EfficientNet-B0 (Calibrated)

Full test-set evaluation using the fitted temperature. All metrics are computed on calibrated probabilities.

In [17]:
test_dataset_effnet = DermatologyDataset(
    csv_path=DATA_TEST_CSV,
    image_dir=DATA_IMAGE_DIR,
    image_column=effnet_bundle['image_column'],
    label_column=effnet_bundle['label_column'],
    class_to_idx=effnet_bundle['class_to_idx'],
    transform=build_transforms(image_size=224, train=False),
)
test_loader_effnet = DataLoader(test_dataset_effnet, batch_size=32, shuffle=False, num_workers=2)

effnet_metrics = evaluate_full(
    model=effnet_bundle['model'],
    test_loader=test_loader_effnet,
    class_names=effnet_bundle['class_names'],
    image_column=effnet_bundle['image_column'],
    temperature=effnet_T,
    device=effnet_bundle['device'],
    output_dir=EFFNET_OUTPUT_DIR,
    tag='effnet_b0_v2',
)


=== Evaluation [effnet_b0_v2] ===
  accuracy: 0.1437
  balanced_accuracy: 0.3077
  macro_f1: 0.1567
  weighted_f1: 0.1782
  top3_accuracy: 0.3949
  macro_roc_auc: 0.7056
  calibrated_nll: 2.1946
  raw_nll → calibrated_nll: 2.6355 → 2.1946
  Saved plots and metrics to: outputs_v2_effnet


## 12. Step 4 — Train ConvNeXt-Tiny V2

ConvNeXt-Tiny is trained under the **identical recipe** as EfficientNet-B0 V2 (same sampler, loss, phases, LR schedule, and early stopping). The only difference is the backbone architecture. This allows a controlled comparison: any metric gap is attributable to capacity and inductive bias rather than training differences.

In [18]:
CONVNEXT_V2_CONFIG = {
    **EFFNET_V2_CONFIG,
    'model_name': 'convnext_tiny',
    'task_name': 'main_class_v2_convnext',
}
print('ConvNeXt-Tiny V2 config ready.')

ConvNeXt-Tiny V2 config ready.


In [19]:
convnext_bundle = train_v2(CONVNEXT_V2_CONFIG, CONVNEXT_OUTPUT_DIR)

Classes (8): ['Infectious Disorders', 'Inflammatory Disorders', 'Keratanisation Disorders', 'Neoplasms and tumors', 'No Definite Diagnosis', 'Other skin disorders', 'Pigmentary Disorders', 'Skin Appendages Disorders']
WeightedRandomSampler active for 3630 train samples
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 276MB/s] 
/tmp/ipykernel_1687/3965782088.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)



=== convnext_tiny | head | epoch 1/15 (global 1) ===


  train_loss=1.0398  val_loss=3.3249  val_macro_f1=0.0782  best=-1.0000
  New best checkpoint saved (macro_f1=0.0782)

=== convnext_tiny | head | epoch 2/15 (global 2) ===


  train_loss=0.7257  val_loss=3.1861  val_macro_f1=0.1173  best=0.0782
  New best checkpoint saved (macro_f1=0.1173)

=== convnext_tiny | head | epoch 3/15 (global 3) ===


...

  train_loss=0.4500  val_loss=3.1106  val_macro_f1=0.1998  best=0.2012

=== convnext_tiny | finetune | epoch 3/20 (global 18) ===


  train_loss=0.4607  val_loss=3.0872  val_macro_f1=0.1982  best=0.2012

=== convnext_tiny | finetune | epoch 4/20 (global 19) ===


  train_loss=0.4593  val_loss=3.0762  val_macro_f1=0.2007  best=0.2012

=== convnext_tiny | finetune | epoch 5/20 (global 20) ===


  train_loss=0.4609  val_loss=3.0522  val_macro_f1=0.2012  best=0.2012

=== convnext_tiny | finetune | epoch 6/20 (global 21) ===


  train_loss=0.4320  val_loss=3.0514  val_macro_f1=0.2003  best=0.2012
  Early stopping after 7 non-improving epochs.

Training complete. Best val macro_f1=0.2012
Best checkpoint: outputs_v2_convnext/checkpoints/best.pt


## 13. Step 5 — Calibrate ConvNeXt-Tiny

Same temperature scaling procedure as EfficientNet-B0. Fitted on validation logits; saved to `outputs_v2_convnext/metrics/temperature.json`.

In [20]:
val_dataset_cnx = DermatologyDataset(
    csv_path=DATA_VAL_CSV,
    image_dir=DATA_IMAGE_DIR,
    image_column=convnext_bundle['image_column'],
    label_column=convnext_bundle['label_column'],
    class_to_idx=convnext_bundle['class_to_idx'],
    transform=build_transforms(image_size=224, train=False),
)
val_loader_cnx = DataLoader(val_dataset_cnx, batch_size=32, shuffle=False, num_workers=2)

cnx_val_logits, cnx_val_targets = collect_val_logits(
    convnext_bundle['model'], val_loader_cnx, convnext_bundle['device']
)
cnx_T, cnx_scaler = calibrate_temperature(
    cnx_val_logits, cnx_val_targets, output_dir=CONVNEXT_OUTPUT_DIR
)
print(f'ConvNeXt-Tiny temperature: {cnx_T:.4f}')

Fitted temperature: 2.0081
ConvNeXt-Tiny temperature: 2.0081


## 14. Step 6 — Evaluate ConvNeXt-Tiny (Calibrated)

Full test-set evaluation using the ConvNeXt-Tiny fitted temperature.

In [21]:
test_dataset_cnx = DermatologyDataset(
    csv_path=DATA_TEST_CSV,
    image_dir=DATA_IMAGE_DIR,
    image_column=convnext_bundle['image_column'],
    label_column=convnext_bundle['label_column'],
    class_to_idx=convnext_bundle['class_to_idx'],
    transform=build_transforms(image_size=224, train=False),
)
test_loader_cnx = DataLoader(test_dataset_cnx, batch_size=32, shuffle=False, num_workers=2)

cnx_metrics = evaluate_full(
    model=convnext_bundle['model'],
    test_loader=test_loader_cnx,
    class_names=convnext_bundle['class_names'],
    image_column=convnext_bundle['image_column'],
    temperature=cnx_T,
    device=convnext_bundle['device'],
    output_dir=CONVNEXT_OUTPUT_DIR,
    tag='convnext_tiny_v2',
)


=== Evaluation [convnext_tiny_v2] ===
  accuracy: 0.1732
  balanced_accuracy: 0.3310
  macro_f1: 0.1841
  weighted_f1: 0.2183
  top3_accuracy: 0.4110
  macro_roc_auc: 0.7267
  calibrated_nll: 2.1566
  raw_nll → calibrated_nll: 2.6276 → 2.1566
  Saved plots and metrics to: outputs_v2_convnext


## 15. Model Comparison

**Selection guidance:**

- **Primary criteria:** `macro_f1` and `balanced_accuracy` — these are robust to class imbalance and directly reflect per-class recall.
- **Supporting criteria:** `top3_accuracy` and `macro_roc_auc` — useful for triage tasks where surfacing the correct class in the top-3 shortlist matters more than top-1 precision.
- **Do NOT select on top-1 accuracy alone** — a model can achieve high top-1 accuracy by over-predicting the majority class while performing poorly on rare but clinically important classes.
- **Tie-breaking:** prefer the backbone with lower `calibrated_nll` (better probability calibration) and lower `temperature` (less post-hoc correction needed).

In [22]:
comparison_metrics = ['macro_f1', 'balanced_accuracy', 'top3_accuracy', 'top5_accuracy', 'macro_roc_auc', 'calibrated_nll', 'temperature']

print(f'{"Metric":<25} {"EfficientNet-B0 V2":>20} {"ConvNeXt-Tiny V2":>20}')
print('-' * 67)
for m in comparison_metrics:
    e_val = effnet_metrics.get(m)
    c_val = cnx_metrics.get(m)
    e_str = f'{e_val:.4f}' if isinstance(e_val, float) else str(e_val)
    c_str = f'{c_val:.4f}' if isinstance(c_val, float) else str(c_val)
    print(f'{m:<25} {e_str:>20} {c_str:>20}')

# Summarise recommendation
print('\nSelection guidance (from spec):')
print('  Primary: macro_f1 + balanced_accuracy')
print('  Supporting: top3_accuracy, macro_roc_auc')
print('  Do NOT select on top-1 accuracy alone.')
print('  Prefer the backbone with lower calibrated_nll (better calibrated) when metrics are tied.')

write_json('comparison_summary.json', {
    'efficientnet_b0_v2': {k: effnet_metrics.get(k) for k in comparison_metrics},
    'convnext_tiny_v2': {k: cnx_metrics.get(k) for k in comparison_metrics},
    'disclaimer': DISCLAIMER_TEXT,
})
print('Saved comparison_summary.json')

Metric                      EfficientNet-B0 V2     ConvNeXt-Tiny V2
-------------------------------------------------------------------
macro_f1                                0.1567               0.1841
balanced_accuracy                       0.3077               0.3310
top3_accuracy                           0.3949               0.4110
top5_accuracy                           0.6632               0.6984
macro_roc_auc                           0.7056               0.7267
calibrated_nll                          2.1946               2.1566
temperature                             2.0514               2.0081

Selection guidance (from spec):
  Primary: macro_f1 + balanced_accuracy
  Supporting: top3_accuracy, macro_roc_auc
  Do NOT select on top-1 accuracy alone.
  Prefer the backbone with lower calibrated_nll (better calibrated) when metrics are tied.
Saved comparison_summary.json


## 16. Optional Inference Demo

Uncomment and edit the cell below to run calibrated top-3 inference on a single image. The `uncertainty_flag` is set to `True` when the top-1 calibrated probability is below 0.45 or when the margin between top-1 and top-2 is less than 0.10 — indicating the model is uncertain and a clinician should review.

**Note:** this requires an actual image file path from your dataset. Do not use this output for clinical decision-making.

In [23]:
# Replace with an actual image path from your dataset.
# result = predict_top3_calibrated(
#     bundle=effnet_bundle,     # or convnext_bundle
#     temperature=effnet_T,     # or cnx_T
#     image_path='data/raw/DATASET/example.jpg',
# )
# import json; print(json.dumps(result, indent=2))

## 17. Save Artifacts To Drive

Copies both backbone output directories and the comparison summary JSON to `MyDrive/derm-opd-triage/outputs_v2/` for persistence across Colab sessions.

In [24]:
from pathlib import Path
import shutil

drive_v2 = Path(DRIVE_PROJECT) / 'outputs_v2'
drive_v2.mkdir(parents=True, exist_ok=True)

for local_dir in [EFFNET_OUTPUT_DIR, CONVNEXT_OUTPUT_DIR]:
    local_path = Path(REPO_DIR) / local_dir
    drive_dest = drive_v2 / local_dir
    if local_path.exists():
        if drive_dest.exists():
            shutil.rmtree(drive_dest)
        shutil.copytree(local_path, drive_dest)
        print(f'Copied {local_path} -> {drive_dest}')

# Save comparison summary
if Path('comparison_summary.json').exists():
    shutil.copy('comparison_summary.json', drive_v2 / 'comparison_summary.json')

print('\nAll artifacts saved to Drive.')
!find {DRIVE_PROJECT}/outputs_v2 -maxdepth 4 -type f | sort

Copied /content/skin-lesion-detect-model/outputs_v2_effnet -> /content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_effnet
Copied /content/skin-lesion-detect-model/outputs_v2_convnext -> /content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext

All artifacts saved to Drive.
/content/drive/MyDrive/derm-opd-triage/outputs_v2/comparison_summary.json
/content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext/checkpoints/best.pt
/content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext/checkpoints/latest.pt
/content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext/metrics/calibration_summary_convnext_tiny_v2.json
/content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext/metrics/classification_report_convnext_tiny_v2.txt
/content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext/metrics/metrics_convnext_tiny_v2.json
/content/drive/MyDrive/derm-opd-triage/outputs_v2/outputs_v2_convnext/metrics/per_class_metrics_conv